データの読み込み

In [4]:
import pandas as pd

df = pd.read_csv("../data/ett.csv", index_col=0, parse_dates=True)

# インデックス範囲の確認
print(f"データの範囲: {df.index.min()} → {df.index.max()}")


データの範囲: 2016-07-01 00:00:00 → 2018-06-26 19:00:00


正規化・スケーリング
・MinMaxScaler を使って0-1の範囲に正規化（LSTMなどのニューラルネットワーク向け）
・StandardScaler を使って平均0, 分散1に標準化（線形回帰やXGBoost向け）

In [5]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# スケーリング（用途に応じて選択）
scaler = MinMaxScaler()  # 0-1正規化
# scaler = StandardScaler()  # 平均0, 分散1の標準化

df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns, index=df.index)

print("\nスケーリング後のデータ:")
print(df_scaled.head())
print(df_scaled.tail())



スケーリング後のデータ:
                         HUFL      HULL      MUFL      MULL      LUFL  \
date                                                                    
2016-07-01 00:00:00  0.615599  0.454943  0.628980  0.467510  0.556576   
2016-07-01 01:00:00  0.612708  0.459449  0.626458  0.464878  0.550279   
2016-07-01 02:00:00  0.601143  0.436920  0.621438  0.459689  0.512595   
2016-07-01 03:00:00  0.599698  0.450437  0.621438  0.462320  0.515693   
2016-07-01 04:00:00  0.605480  0.450437  0.626458  0.467510  0.521990   

                         LULL        OT  
date                                     
2016-07-01 00:00:00  0.613765  0.691018  
2016-07-01 01:00:00  0.620783  0.636233  
2016-07-01 02:00:00  0.586144  0.636233  
2016-07-01 03:00:00  0.599955  0.581468  
2016-07-01 04:00:00  0.599955  0.519656  
                         HUFL      HULL      MUFL      MULL      LUFL  \
date                                                                    
2018-06-26 15:00:00  0.453765  0.5

時系列特徴量の作成
・移動平均・ラグ特徴量（過去の値を追加）
・変化率特徴量（直前の値との差分）

・時間情報を追加（曜日、時間帯、季節性をキャプチャ）

In [6]:
# 時間（0~23）
df_scaled["hour"] = df_scaled.index.hour  

# ピーク時間（15:00〜17:00）フラグ
df_scaled["is_peak_hour"] = df_scaled["hour"].isin([15, 16, 17]).astype(int)

# 夜間（0:00〜5:59）フラグ
df_scaled["is_night"] = (df_scaled["hour"] < 6).astype(int)

# 曜日（0=月曜日, 6=日曜日）
df_scaled["dayofweek"] = df_scaled.index.dayofweek  

# 月（1~12）
df_scaled["month"] = df_scaled.index.month  

# 確認
print("追加された特徴量のサンプル:")
print(df_scaled[["hour", "is_peak_hour", "is_night", "dayofweek", "month"]].head())



追加された特徴量のサンプル:
                     hour  is_peak_hour  is_night  dayofweek  month
date                                                               
2016-07-01 00:00:00     0             0         1          4      7
2016-07-01 01:00:00     1             0         1          4      7
2016-07-01 02:00:00     2             0         1          4      7
2016-07-01 03:00:00     3             0         1          4      7
2016-07-01 04:00:00     4             0         1          4      7


移動平均・ラグ特徴量（過去の値を追加）

In [7]:
df_scaled["OT_lag1"] = df_scaled["OT"].shift(1)  # 1時間前のOT
df_scaled["OT_lag2"] = df_scaled["OT"].shift(2)  # 2時間前のOT
df_scaled["OT_lag3"] = df_scaled["OT"].shift(3)  # 3時間前のOT
df_scaled["OT_roll_mean_3"] = df_scaled["OT"].rolling(window=3).mean()  # 直近3時間の平均
df_scaled["OT_roll_mean_6"] = df_scaled["OT"].rolling(window=6).mean()  # 直近6時間の平均
df_scaled["OT_roll_mean_12"] = df_scaled["OT"].rolling(window=12).mean()  # 直近12時間の平均

# 確認
print(df_scaled[["OT", "OT_roll_mean_3", "OT_roll_mean_6", "OT_roll_mean_12", "OT_lag1", "OT_lag2", "OT_lag3"]].head(10))


                           OT  OT_roll_mean_3  OT_roll_mean_6  \
date                                                            
2016-07-01 00:00:00  0.691018             NaN             NaN   
2016-07-01 01:00:00  0.636233             NaN             NaN   
2016-07-01 02:00:00  0.636233        0.654495             NaN   
2016-07-01 03:00:00  0.581468        0.617978             NaN   
2016-07-01 04:00:00  0.519656        0.579119             NaN   
2016-07-01 05:00:00  0.504203        0.535109        0.594802   
2016-07-01 06:00:00  0.536506        0.520122        0.569050   
2016-07-01 07:00:00  0.543534        0.528081        0.553600   
2016-07-01 08:00:00  0.514046        0.531362        0.533236   
2016-07-01 09:00:00  0.429772        0.495784        0.507953   

                     OT_roll_mean_12   OT_lag1   OT_lag2   OT_lag3  
date                                                                
2016-07-01 00:00:00              NaN       NaN       NaN       NaN  
2016-07-01 0

季節情報
外気温データがない場合でも、季節を考慮することで間接的に影響を加味できる。

In [8]:
# 夏（7月・8月）のフラグ
df_scaled["is_summer"] = df_scaled["month"].isin([7, 8]).astype(int)

# 冬（12月・1月）のフラグ
df_scaled["is_winter"] = df_scaled["month"].isin([12, 1]).astype(int)

# 月ごとの平均温度を計算（元のOTのスケールを維持）
monthly_avg_temp = df_scaled.groupby("month")["OT"].transform("mean")

# 月ごとの温度異常度を計算（各データ点がその月の平均からどれだけずれているか）
df_scaled["monthly_temp_anomaly"] = df_scaled["OT"] - monthly_avg_temp

# 確認
print("追加された特徴量のサンプル:")
print(df_scaled[["month", "is_summer", "is_winter", "monthly_temp_anomaly"]].head())


追加された特徴量のサンプル:
                     month  is_summer  is_winter  monthly_temp_anomaly
date                                                                  
2016-07-01 00:00:00      7          1          0              0.062109
2016-07-01 01:00:00      7          1          0              0.007324
2016-07-01 02:00:00      7          1          0              0.007324
2016-07-01 03:00:00      7          1          0             -0.047440
2016-07-01 04:00:00      7          1          0             -0.109253


前処理 & 特徴量エンジニアリング後のデータを保存

In [9]:
df_scaled.to_csv("../data/preprocessed_data.csv", index=True)  # 前処理済みデータを保存


まとめ
📌 preprocessing.ipynb でやったこと ✅ データのスケーリング（標準化）
✅ 時間関連の特徴量追加（hour, dayofweek, month）
✅ ラグ特徴量の作成（OT_lag1, OT_lag2, OT_lag3）
✅ 移動平均（OT_roll_mean_3, OT_roll_mean_6, OT_roll_mean_12）
✅ 高負荷フラグの作成（high_load）
✅ 季節情報フラグ（is_summer, is_winter）
✅ 最終的なデータを preprocessed_data.csv に保存

